<a href="https://colab.research.google.com/github/pradhapmoorthi/CVND/blob/BeamSearch/image_captioning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🖼️ Image Captioning — Unified Pipeline (Global/Spatial Attention + Greedy/Beam)

This single notebook **merges** the earlier full pipeline with a **second variant** that adds **spatial attention over CNN feature maps** and **beam search decoding**. Choose behavior with `CFG`:

- `CFG['use_spatial_attention']`: `False` (global feature attention) or `True` (spatial attention over H×W)
- `CFG['decode']`: `'greedy'` or `'beam'`
- `CFG['beam_size']`: beam width when `decode='beam'` (inference only)

**Datasets supported:** Flickr8k (default) and COCO 2017 (optional)

> Tip: Training is compute‑intensive. Prefer a GPU runtime.


## 1) Setup & Configuration

In [1]:
import os, re, io, json, math, time, random
from pathlib import Path
from typing import List, Dict, Tuple

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive



## 2) Data Preparation
Ensure the dataset files are arranged as noted in the config. This notebook does **not** auto‑download datasets.


In [25]:
import os, re, io, json, math, time, random
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision
from PIL import Image
import matplotlib.pyplot as plt

try:
    from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
    nltk_ok = True
except Exception:
    nltk_ok = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

SPECIAL_TOKENS = {'pad':'<pad>', 'bos':'<bos>', 'eos':'<eos>', 'unk':'<unk>'}

CFG = {
    'dataset': 'COCO',          # 'Flickr8k' or 'COCO'
    'data_root': '/data',

    # Flickr8k expected structure:
    #   ./data/Flickr8k/images/
    #   ./data/Flickr8k/captions/Flickr8k.token.txt
    #   ./data/Flickr8k/captions/Flickr_8k.trainImages.txt
    #   ./data/Flickr8k/captions/Flickr_8k.devImages.txt
    #   ./data/Flickr8k/captions/Flickr_8k.testImages.txt

    # COCO expected structure:
    #   ./data/COCO/train2017/*.jpg
    #   ./data/COCO/val2017/*.jpg
    #   ./data/COCO/annotations/captions_train2017.json
    #   ./data/COCO/annotations/captions_val2017.json

    'min_freq': 5,
    'max_len': 20,
    'batch_size': 64,
    'num_workers': 2,

    # Model & training
    'use_spatial_attention': True,   # <— toggle here
    'encoder_cnn': 'resnet50',
    'embed_dim': 256,
    'hidden_dim': 512,
    'attention_dim': 256,
    'dropout': 0.3,

    'epochs': 10,            # increase for real training
    'lr': 3e-4,
    'clip': 1.0,
    'teacher_forcing': 0.5,

    # Decoding
    'decode': 'beam',       # 'greedy' or 'beam'
    'beam_size': 3,

    'save_dir': '/content/drive/MyDrive/image_captioning_checkpoints',
    'exp_name': 'captioning_unified',
    'fp16': True,
}

os.makedirs(CFG['save_dir'], exist_ok=True)

Device: cuda


In [4]:
import os, re, io, json, math, time, random
from pathlib import Path
from typing import List, Dict, Tuple

SOURCE_TRAIN_ZIP_PATH = '/content/drive/MyDrive/datasets/COCO/train2017.zip' # <--- UPDATE THESE PATHS
SOURCE_VAL_ZIP_PATH = '/content/drive/MyDrive/datasets/COCO/val2017.zip'
SOURCE_ANN_ZIP_PATH = '/content/drive/MyDrive/datasets/COCO/annotations_trainval2017.zip'

DEST_DIR = Path('/data/COCO')

# Ensure the base destination directory exists
DEST_DIR.mkdir(parents=True, exist_ok=True)
print(f"Ensured base directory '{DEST_DIR}' exists.")

def copy_and_extract_zip(source_path, dest_dir, extract_to_subdir=None):
    if not Path(source_path).exists():
        print(f"Source file '{source_path}' does not exist. Skipping.")
        return

    zip_filename = Path(source_path).name
    dest_zip_path = dest_dir / zip_filename

    if not dest_zip_path.exists():
        print(f"Copying {source_path} to {dest_zip_path}...")
        !cp "{source_path}" "{dest_zip_path}"
        print("Copy complete.")
    else:
        print(f"'{dest_zip_path}' already exists. Skipping copy.")

    if dest_zip_path.exists():
        print(f"'{dest_zip_path}' successfully copied.")

        extract_target_dir = dest_dir / (extract_to_subdir if extract_to_subdir else zip_filename.split('.')[0])

        # Always force re-extraction if the directory exists for COCO, as previous heuristics might fail.
        # This is a more aggressive approach to ensure a clean slate after issues.
        force_re_extract = False
        if extract_target_dir.is_dir():
            print(f"Directory '{extract_target_dir}' found. Forcing re-extraction for reliability...")
            force_re_extract = True

        if force_re_extract:
            print(f"Removing '{extract_target_dir}' for re-extraction.")
            !rm -rf "{extract_target_dir}" # Use -rf to ensure removal of non-empty directory

        if not extract_target_dir.exists():
            extract_target_dir.mkdir(parents=True, exist_ok=True)
            print(f"Created extraction directory: {extract_target_dir}")

            print(f"Extracting {dest_zip_path} to {extract_target_dir}...")
            !unzip -q "{dest_zip_path}" -d "{extract_target_dir}"
            print("Extraction complete.")

            # Custom handling for COCO annotations if they are nested
            if extract_to_subdir == 'annotations':
                nested_annotations_path = extract_target_dir / 'annotations'
                if nested_annotations_path.is_dir() and any(nested_annotations_path.iterdir()):
                    print(f"Found nested annotations directory: {nested_annotations_path}. Moving contents...")
                    # Use rsync to move contents more robustly
                    !rsync -a "{nested_annotations_path}/" "{extract_target_dir}"/
                    !rm -rf "{nested_annotations_path}" # Use rm -rf here
                    print("Moved nested annotations contents.")
            # Custom handling for COCO images if they are nested (e.g., train2017.zip extracts to train2017/train2017)
            elif extract_to_subdir in ['train2017', 'val2017']:
                nested_image_path = extract_target_dir / extract_to_subdir
                if nested_image_path.is_dir() and any(nested_image_path.iterdir()):
                    print(f"Found nested image directory: {nested_image_path}. Moving contents...")
                    # Use rsync to move contents more robustly
                    !rsync -a "{nested_image_path}/" "{extract_target_dir}"/
                    !rm -rf "{nested_image_path}" # Use rm -rf here
                    print("Moved nested image contents.")

            # Optional: Remove the copied zip file to save space
            print(f"Removing {dest_zip_path}...")
            !rm "{dest_zip_path}"
            print("Zip file removed.")
        else:
            # This 'else' block should ideally not be reached if force_re_extract is True
            print(f"Extraction directory '{extract_target_dir}' already exists, but was supposed to be removed. Something went wrong.")
    else:
        print(f"Error: '{dest_zip_path}' not found after copy. Please ensure {source_path} is correct and try again.")


copy_and_extract_zip(SOURCE_TRAIN_ZIP_PATH, DEST_DIR, 'train2017')
copy_and_extract_zip(SOURCE_VAL_ZIP_PATH, DEST_DIR, 'val2017')
copy_and_extract_zip(SOURCE_ANN_ZIP_PATH, DEST_DIR, 'annotations')

# Verify the dataset structure
print("\nVerifying COCO dataset structure:")
print(f"Is {DEST_DIR / 'train2017'} directory present? { (DEST_DIR / 'train2017').is_dir() }")
print(f"Is {DEST_DIR / 'val2017'} directory present? { (DEST_DIR / 'val2017').is_dir() }")
print(f"Is {DEST_DIR / 'annotations'} directory present? { (DEST_DIR / 'annotations').is_dir() }")
print(f"Is {DEST_DIR / 'annotations' / 'captions_train2017.json'} present? { (DEST_DIR / 'annotations' / 'captions_train2017.json').is_file() }")
print(f"Is {DEST_DIR / 'annotations' / 'captions_val2017.json'} present? { (DEST_DIR / 'annotations' / 'captions_val2017.json').is_file() }")

Ensured base directory '/data/COCO' exists.
Copying /content/drive/MyDrive/datasets/COCO/train2017.zip to /data/COCO/train2017.zip...
Copy complete.
'/data/COCO/train2017.zip' successfully copied.
Created extraction directory: /data/COCO/train2017
Extracting /data/COCO/train2017.zip to /data/COCO/train2017...
Extraction complete.
Found nested image directory: /data/COCO/train2017/train2017. Moving contents...
Moved nested image contents.
Removing /data/COCO/train2017.zip...
Zip file removed.
Copying /content/drive/MyDrive/datasets/COCO/val2017.zip to /data/COCO/val2017.zip...
Copy complete.
'/data/COCO/val2017.zip' successfully copied.
Created extraction directory: /data/COCO/val2017
Extracting /data/COCO/val2017.zip to /data/COCO/val2017...
Extraction complete.
Found nested image directory: /data/COCO/val2017/val2017. Moving contents...
Moved nested image contents.
Removing /data/COCO/val2017.zip...
Zip file removed.
Copying /content/drive/MyDrive/datasets/COCO/annotations_trainval201

## 3) Vocabulary & Tokenization

In [5]:

class Vocabulary:
    def __init__(self, min_freq=5):
        self.min_freq = min_freq
        self.freqs = {}
        self.stoi = {}
        self.itos = []

    @staticmethod
    def tokenize(text: str) -> List[str]:
        text = text.lower()
        return re.findall(r"[a-zA-Z]+|\d+|[^\s\w]", text)

    def build(self, sentences: List[str]):
        for s in sentences:
            for tok in self.tokenize(s):
                self.freqs[tok] = self.freqs.get(tok, 0) + 1
        self.itos = [SPECIAL_TOKENS['pad'], SPECIAL_TOKENS['bos'], SPECIAL_TOKENS['eos'], SPECIAL_TOKENS['unk']]
        self.stoi = {tok: i for i, tok in enumerate(self.itos)}
        for tok, f in sorted(self.freqs.items(), key=lambda x: (-x[1], x[0])):
            if f >= self.min_freq and tok not in self.stoi:
                self.stoi[tok] = len(self.itos)
                self.itos.append(tok)
        print(f"Built vocab size: {len(self)}")

    def __len__(self):
        return len(self.itos)

    def numericalize(self, tokens: List[str]) -> List[int]:
        return [self.stoi.get(t, self.stoi[SPECIAL_TOKENS['unk']]) for t in tokens]

    def denumericalize(self, ids: List[int]) -> List[str]:
        return [self.itos[i] for i in ids]


def save_vocab(vocab: Vocabulary, path: str):
    obj = {'min_freq': vocab.min_freq, 'itos': vocab.itos}
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f)


def load_vocab(path: str) -> Vocabulary:
    with open(path, 'r', encoding='utf-8') as f:
        obj = json.load(f)
    v = Vocabulary(min_freq=obj['min_freq'])
    v.itos = obj['itos']
    v.stoi = {tok: i for i, tok in enumerate(v.itos)}
    return v


## 4) Dataset, Transforms, & DataLoader

In [6]:

class Flickr8kCaptions:
    def __init__(self, root: str):
        root = Path(root)
        self.images_dir = root / 'images'
        self.captions_dir = root / 'captions'
        self.token_file = self.captions_dir / 'Flickr8k.token.txt'
        self.train_file = self.captions_dir / 'Flickr_8k.trainImages.txt'
        self.dev_file   = self.captions_dir / 'Flickr_8k.devImages.txt'
        self.test_file  = self.captions_dir / 'Flickr_8k.testImages.txt'
        assert self.token_file.exists(), f"Missing {self.token_file}"
        self.image2caps = {}
        with open(self.token_file, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('	')
                if len(parts) != 2: continue
                img_id, cap = parts
                img = img_id.split('#')[0]
                self.image2caps.setdefault(img, []).append(cap)
        def read_list(p):
            with open(p, 'r', encoding='utf-8') as f:
                return [x.strip() for x in f if x.strip()]
        self.train_imgs = read_list(self.train_file)
        self.val_imgs   = read_list(self.dev_file)
        self.test_imgs  = read_list(self.test_file)

class COCOV2:
    def __init__(self, root: str):
        root = Path(root)
        self.train_dir = root / 'train2017'
        self.val_dir   = root / 'val2017'
        ann_dir = root / 'annotations'
        with open(ann_dir / 'captions_train2017.json', 'r') as f:
            train_ann = json.load(f)
        with open(ann_dir / 'captions_val2017.json', 'r') as f:
            val_ann = json.load(f)
        def build(ann, img_dir):
            id2file = {img['id']: img['file_name'] for img in ann['images']}
            file2caps = {}
            for a in ann['annotations']:
                fn = id2file[a['image_id']]
                file2caps.setdefault(fn, []).append(a['caption'])
            files = [f for f in file2caps.keys() if (img_dir / f).exists()]
            return img_dir, file2caps, files
        self.train_dir, self.train_caps, self.train_files = build(train_ann, self.train_dir)
        self.val_dir,   self.val_caps,   self.val_files   = build(val_ann, self.val_dir)

class ImageCaptionDataset(Dataset):
    def __init__(self, dataset_name: str, split: str, cfg: Dict, vocab: Vocabulary=None):
        self.dataset_name = dataset_name
        self.split = split
        self.cfg = cfg
        self.vocab = vocab
        self.max_len = cfg['max_len']

        pairs = []
        if dataset_name.lower() == 'flickr8k':
            flickr = Flickr8kCaptions(Path(cfg['data_root']) / 'Flickr8k')
            files = flickr.train_imgs if split=='train' else (flickr.val_imgs if split=='val' else flickr.test_imgs)
            for fn in files:
                for cap in flickr.image2caps.get(fn, []):
                    pairs.append((flickr.images_dir / fn, cap))
        elif dataset_name.lower() == 'coco':
            coco = COCOV2(Path(cfg['data_root']) / 'COCO')
            if split=='train':
                files, caps_map, base = coco.train_files, coco.train_caps, coco.train_dir
            else:
                files, caps_map, base = coco.val_files, coco.val_caps, coco.val_dir
            for fn in files:
                for cap in caps_map.get(fn, []):
                    pairs.append((base / fn, cap))
        else:
            raise ValueError('Unknown dataset: ' + dataset_name)
        self.pairs = pairs
        print(f"Loaded {len(self.pairs)} pairs for {dataset_name} [{split}]")

        if split=='train':
            self.tfms = transforms.Compose([
                transforms.Resize((256,256)),
                transforms.RandomCrop((224,224)),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
            ])
        else:
            self.tfms = transforms.Compose([
                transforms.Resize((224,224)),
                transforms.CenterCrop((224,224)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
            ])

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        path, caption = self.pairs[idx]
        img = Image.open(path).convert('RGB')
        img = self.tfms(img)
        tokens = [SPECIAL_TOKENS['bos']] + self.vocab.tokenize(caption) + [SPECIAL_TOKENS['eos']]
        ids = self.vocab.numericalize(tokens)
        if len(ids) > self.max_len:
            ids = ids[:self.max_len]
            if ids[-1] != self.vocab.stoi[SPECIAL_TOKENS['eos']]:
                ids[-1] = self.vocab.stoi[SPECIAL_TOKENS['eos']]
        length = len(ids)
        return img, torch.tensor(ids, dtype=torch.long), length

def pad_collate(batch):
    imgs, seqs, lens = zip(*batch)
    imgs = torch.stack(imgs, dim=0)
    max_len = max(lens)
    pad_id = 0
    padded = torch.full((len(seqs), max_len), pad_id, dtype=torch.long)
    for i, s in enumerate(seqs):
        padded[i, :len(s)] = s
    lens = torch.tensor(lens, dtype=torch.long)
    return imgs, padded, lens


### Build/Load Vocabulary

In [7]:

vocab_path = Path(CFG['save_dir']) / f"{CFG['dataset'].lower()}_vocab.json"
if vocab_path.exists():
    vocab = load_vocab(str(vocab_path))
    print(f"Loaded vocab from {vocab_path} (size={len(vocab)})")
else:
    corpus = []
    if CFG['dataset'].lower() == 'flickr8k':
        flickr = Flickr8kCaptions(Path(CFG['data_root']) / 'Flickr8k')
        for fn in flickr.train_imgs:
            corpus.extend(flickr.image2caps.get(fn, []))
    else:
        coco = COCOV2(Path(CFG['data_root']) / 'COCO')
        for fn in coco.train_files:
            corpus.extend(coco.train_caps.get(fn, []))
    vocab = Vocabulary(min_freq=CFG['min_freq'])
    vocab.build(corpus)
    save_vocab(vocab, str(vocab_path))
    print(f"Saved vocab to {vocab_path}")


Built vocab size: 10217
Saved vocab to /content/drive/MyDrive/image_captioning_checkpoints/coco_vocab.json


### DataLoaders

In [8]:
train_ds = ImageCaptionDataset(CFG['dataset'], 'train', CFG, vocab)
val_ds   = ImageCaptionDataset(CFG['dataset'], 'val',   CFG, vocab)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=CFG['num_workers'], collate_fn=pad_collate, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], collate_fn=pad_collate, pin_memory=True)

len(train_loader), len(val_loader)

Loaded 591753 pairs for COCO [train]
Loaded 25014 pairs for COCO [val]


(9247, 391)

## 5) Models: Encoder/Decoder + Attention (Global & Spatial)

In [10]:
import os, re, io, json, math, time, random
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision
from PIL import Image
import matplotlib.pyplot as plt

try:
    from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
    nltk_ok = True
except Exception:
    nltk_ok = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

SPECIAL_TOKENS = {'pad':'<pad>', 'bos':'<bos>', 'eos':'<eos>', 'unk':'<unk>'}

# -------- Global feature encoder & attention --------
class EncoderCNN_Global(nn.Module):
    def __init__(self, cnn_name='resnet50', embed_dim=256, train_backbone=False):
        super().__init__()
        if cnn_name == 'resnet50':
            backbone = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
            modules = list(backbone.children())[:-1]
            self.cnn = nn.Sequential(*modules)
            feat_dim = backbone.fc.in_features
        else:
            raise ValueError('Unsupported CNN: ' + cnn_name)
        for p in self.cnn.parameters():
            p.requires_grad = train_backbone
        self.fc = nn.Linear(feat_dim, embed_dim)
        self.bn = nn.BatchNorm1d(embed_dim)

    def forward(self, images):
        feats = self.cnn(images).flatten(1)
        feats = F.relu(self.bn(self.fc(feats)))  # (B,E)
        return feats

class BahdanauAttention_Global(nn.Module):
    def __init__(self, enc_dim, dec_hidden, attn_dim):
        super().__init__()
        self.W = nn.Linear(enc_dim, attn_dim)
        self.U = nn.Linear(dec_hidden, attn_dim)
        self.v = nn.Linear(attn_dim, 1)
    def forward(self, enc_out, hidden):
        score = self.v(torch.tanh(self.W(enc_out) + self.U(hidden)))  # (B,1)
        alpha = torch.softmax(score, dim=1)                            # (B,1)
        context = alpha * enc_out                                      # (B,E)
        return context, alpha

class Decoder_Global(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, attn_dim, dropout=0.3):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.attn  = BahdanauAttention_Global(embed_dim, hidden_dim, attn_dim)
        self.lstm  = nn.LSTMCell(embed_dim + embed_dim, hidden_dim)
        self.fc    = nn.Linear(hidden_dim, vocab_size)
        self.drop  = nn.Dropout(dropout)
    def forward(self, enc_feat, captions):
        B,T = captions.size()
        h = captions.new_zeros((B, self.lstm.hidden_size), dtype=torch.float).to(captions.device)
        c = captions.new_zeros((B, self.lstm.hidden_size), dtype=torch.float).to(captions.device)
        inp = self.embed(captions[:,0])
        outs=[]
        for t in range(1,T):
            ctx,_ = self.attn(enc_feat, h)
            h,c = self.lstm(torch.cat([inp, ctx], dim=1), (h,c))
            logits = self.fc(self.drop(h))
            outs.append(logits.unsqueeze(1))
            inp = self.embed(captions[:,t])
        return torch.cat(outs, dim=1)
    def greedy_decode(self, enc_feat, bos_id, eos_id, max_len=20):
        B = enc_feat.size(0)
        h = enc_feat.new_zeros((B, self.lstm.hidden_size))
        c = enc_feat.new_zeros((B, self.lstm.hidden_size))
        x = torch.full((B,), bos_id, dtype=torch.long, device=enc_feat.device)
        emb = self.embed(x)
        seqs=[]
        for _ in range(max_len):
            ctx,_ = self.attn(enc_feat, h)
            h,c = self.lstm(torch.cat([emb, ctx], dim=1), (h,c))
            logit = self.fc(h)
            x = logit.argmax(-1)
            seqs.append(x)
            emb = self.embed(x)
        out=[]
        for b in range(B):
            toks=[]
            for t in seqs:
                tok = t[b].item()
                if tok==eos_id: break
                toks.append(tok)
            out.append(toks)
        return out

# -------- Spatial feature encoder & attention --------
class EncoderCNN_Spatial(nn.Module):
    def __init__(self):
        super().__init__()
        m = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
        self.cnn = nn.Sequential(*list(m.children())[:-2])  # Bx2048x7x7
        self.adapt = nn.Conv2d(2048, 512, kernel_size=1)
    def forward(self, images):
        fmap = self.cnn(images)     # B,2048,7,7
        fmap = self.adapt(fmap)     # B,512,7,7
        B,C,H,W = fmap.shape
        seq = fmap.permute(0,2,3,1).contiguous().view(B, H*W, C)  # B,T(=49),512
        return seq, (H,W)

class SpatialAttention(nn.Module):
    def __init__(self, feat_dim, hidden_dim):
        super().__init__()
        self.W = nn.Linear(feat_dim, hidden_dim)
        self.U = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1)
    def forward(self, feats, hidden):
        # feats: B,T,C; hidden: B,H
        score = self.v(torch.tanh(self.W(feats) + self.U(hidden).unsqueeze(1)))  # B,T,1
        alpha = torch.softmax(score, dim=1)                                       # B,T,1
        ctx = (alpha * feats).sum(1)                                              # B,C
        return ctx, alpha

class Decoder_Spatial(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512, feat_dim=512, dropout=0.3):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.attn  = SpatialAttention(feat_dim, hidden_dim)
        self.lstm  = nn.LSTMCell(embed_dim + feat_dim, hidden_dim)
        self.fc    = nn.Linear(hidden_dim, vocab_size)
        self.drop  = nn.Dropout(dropout)
        self.hidden_dim = hidden_dim
    def forward(self, feats, captions):
        B,T = captions.size()
        h = feats.new_zeros((B, self.hidden_dim))
        c = feats.new_zeros((B, self.hidden_dim))
        inp = self.embed(captions[:,0])
        outs=[]
        for t in range(1,T):
            ctx,_ = self.attn(feats, h)
            h,c = self.lstm(torch.cat([inp, ctx], dim=1), (h,c))
            logits = self.fc(self.drop(h))
            outs.append(logits.unsqueeze(1))
            inp = self.embed(captions[:,t])
        return torch.cat(outs, dim=1)
    def greedy_decode(self, feats, bos_id, eos_id, max_len=20):
        B = feats.size(0)
        h = feats.new_zeros((B, self.hidden_dim))
        c = feats.new_zeros((B, self.hidden_dim))
        x = torch.full((B,), bos_id, dtype=torch.long, device=feats.device)
        emb = self.embed(x)
        seqs=[]; alphas=[]
        for _ in range(max_len):
            ctx,alpha = self.attn(feats, h)
            h,c = self.lstm(torch.cat([emb, ctx], dim=1), (h,c))
            logit = self.fc(h)
            x = logit.argmax(-1)
            seqs.append(x)
            alphas.append(alpha.squeeze(-1))  # B,T
            emb = self.embed(x)
        # stop at eos per sample
        out=[]
        for b in range(B):
            toks=[]
            for t in seqs:
                tok=t[b].item()
                if tok==eos_id: break
                toks.append(tok)
            out.append(toks)
        return out, alphas  # list of tokens per sample, and list of alpha tensors per step
    def beam_search(self, feats, bos_id, eos_id, beam=3, max_len=20):
        # NOTE: supports B==1 for simplicity
        assert feats.size(0) == 1, 'Beam search currently supports batch size 1.'
        h = feats.new_zeros((1, self.hidden_dim))
        c = feats.new_zeros((1, self.hidden_dim))
        sequences = [([bos_id], 0.0, h, c)]  # (tokens, logprob, h, c)
        for _ in range(max_len):
            new_list = []
            for toks,score,hx,cx in sequences:
                if toks[-1] == eos_id:
                    new_list.append((toks, score, hx, cx))
                    continue
                x = torch.tensor([toks[-1]], device=feats.device)
                emb = self.embed(x)
                ctx,_ = self.attn(feats, hx)
                hx, cx = self.lstm(torch.cat([emb, ctx], dim=1), (hx, cx))
                logits = self.fc(hx)
                logprobs = F.log_softmax(logits, dim=-1)
                topk = torch.topk(logprobs, beam)
                for i in range(beam):
                    tok = int(topk.indices[0, i].item())
                    sc  = float(score + topk.values[0, i].item())
                    new_list.append((toks + [tok], sc, hx.clone(), cx.clone()))
            # prune
            new_list.sort(key=lambda x: x[1], reverse=True)
            sequences = new_list[:beam]
        best = sequences[0][0]
        # strip BOS and cut at EOS
        out=[]
        for t in best[1:]:
            if t==eos_id: break
            out.append(t)
        return out


Device: cuda


### Build Models per Config

In [20]:
# Model factory
if CFG['use_spatial_attention']:
    encoder = EncoderCNN_Spatial().to(device)
    decoder = Decoder_Spatial(
        vocab_size=len(vocab),
        embed_dim=CFG['embed_dim'],
        hidden_dim=CFG['hidden_dim'],
        feat_dim=512,
        dropout=CFG['dropout']).to(device)
else:
    encoder = EncoderCNN_Global(CFG['encoder_cnn'], CFG['embed_dim']).to(device)
    decoder = Decoder_Global(
        vocab_size=len(vocab),
        embed_dim=CFG['embed_dim'],
        hidden_dim=CFG['hidden_dim'],
        attn_dim=CFG['attention_dim'],
        dropout=CFG['dropout']).to(device)

params = list(decoder.parameters()) + [p for p in encoder.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(params, lr=CFG['lr'])
criterion = nn.CrossEntropyLoss(ignore_index=0)
scaler = torch.amp.GradScaler('cuda', enabled=CFG['fp16'])

ckpt_path = Path(CFG['save_dir']) / f"{CFG['exp_name']}.pt"

## 6) Training & Evaluation (BLEU‑4)

In [ ]:
def train_one_epoch(epoch):
    encoder.train(); decoder.train()
    total = 0.0
    step_group_start_time = time.time() # Initialize timer for step groups
    for i,(imgs,caps,lens) in enumerate(train_loader):
        imgs, caps = imgs.to(device), caps.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=CFG['fp16']):
            if CFG['use_spatial_attention']:
                feats, _ = encoder(imgs)
                logits = decoder(feats, caps)
            else:
                feat = encoder(imgs)
                logits = decoder(feat, caps)
            targets = caps[:,1:logits.size(1)+1]
            loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        scaler.scale(loss).backward()
        nn.utils.clip_grad_norm_(params, CFG['clip'])
        scaler.step(optimizer)
        scaler.update()
        total += loss.item()
        if (i+1)%50==0:
            elapsed_time = time.time() - step_group_start_time # Calculate elapsed time for 50 steps
            print(f"epoch {epoch} step {i+1}/{len(train_loader)} loss {total/(i+1):.4f} (last 50 steps took {elapsed_time:.2f}s)")
            step_group_start_time = time.time() # Reset timer for next 50 steps
    return total/max(1,len(train_loader))


def evaluate_bleu(sample_limit=1000):
    encoder.eval(); decoder.eval()
    refs, hyps = [], []
    with torch.no_grad():
        count=0
        for imgs, caps, lens in val_loader:
            imgs = imgs.to(device)
            if CFG['use_spatial_attention']:
                feats,_ = encoder(imgs)
                out_ids,_ = decoder.greedy_decode(feats, vocab.stoi[SPECIAL_TOKENS['bos']], vocab.stoi[SPECIAL_TOKENS['eos']], max_len=CFG['max_len'])
            else:
                feat = encoder(imgs)
                out_ids = decoder.greedy_decode(feat, vocab.stoi[SPECIAL_TOKENS['bos']], vocab.stoi[SPECIAL_TOKENS['eos']], max_len=CFG['max_len'])
            for b in range(len(out_ids)):
                tgt_ids = caps[b].tolist()
                # strip bos/eos/pad
                try: bos = tgt_ids.index(vocab.stoi[SPECIAL_TOKENS['bos']])
                except ValueError: bos=0
                eos = tgt_ids.index(vocab.stoi[SPECIAL_TOKENS['eos']]) if vocab.stoi[SPECIAL_TOKENS['eos']] in tgt_ids else len(tgt_ids)
                ref = vocab.denumericalize(tgt_ids[bos+1:eos])
                hyp = vocab.denumericalize(out_ids[b])
                if ref and hyp:
                    refs.append([ref])
                    hyps.append(hyp)
            count += len(out_ids)
            if count>=sample_limit: break
    if nltk_ok and hyps:
        smoothie = SmoothingFunction().method4
        return corpus_bleu(refs, hyps, smoothing_function=smoothie)
    return 0.0

best_bleu = 0.0
patience = 5
patience_counter = 0

# Ensure the save directory for checkpoints exists
os.makedirs(CFG['save_dir'], exist_ok=True)
ckpt_path = Path(CFG['save_dir']) / f"{CFG['exp_name']}.pt"

overall_start_time = time.time() # Start overall training timer

for epoch in range(1, CFG['epochs']+1):
    epoch_start_time = time.time() # Start epoch timer
    tr = train_one_epoch(epoch)
    bl = evaluate_bleu(sample_limit=1000)
    epoch_duration = time.time() - epoch_start_time # Calculate epoch duration
    print(f"Epoch {epoch}: loss={tr:.4f} BLEU-4={bl:.4f} (duration: {epoch_duration:.2f}s)")

    if bl > best_bleu:
        best_bleu = bl
        patience_counter = 0
        torch.save({'encoder': encoder.state_dict(), 'decoder': decoder.state_dict(), 'cfg': CFG}, ckpt_path)
        print('Saved best model to ->', ckpt_path)
    else:
        patience_counter += 1
        print(f"BLEU-4 did not improve. Patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print(f"Early stopping triggered after {patience} epochs without improvement.")
            break

total_training_duration = time.time() - overall_start_time # Calculate total training duration
print(f"Total training duration: {total_training_duration:.2f}s")

/tmp/ipython-input-2864570350.py:8: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=CFG['fp16']):


epoch 1 step 50/9247 loss 3.9282 (last 50 steps took 16.87s)
epoch 1 step 100/9247 loss 3.9056 (last 50 steps took 17.47s)
epoch 1 step 150/9247 loss 3.8987 (last 50 steps took 17.11s)
epoch 1 step 200/9247 loss 3.8879 (last 50 steps took 16.26s)
epoch 1 step 250/9247 loss 3.8637 (last 50 steps took 16.00s)
epoch 1 step 300/9247 loss 3.8437 (last 50 steps took 16.13s)
epoch 1 step 350/9247 loss 3.8274 (last 50 steps took 16.33s)
epoch 1 step 400/9247 loss 3.8051 (last 50 steps took 16.52s)
epoch 1 step 450/9247 loss 3.7883 (last 50 steps took 16.55s)
epoch 1 step 500/9247 loss 3.7690 (last 50 steps took 16.50s)
epoch 1 step 550/9247 loss 3.7572 (last 50 steps took 16.54s)
epoch 1 step 600/9247 loss 3.7478 (last 50 steps took 16.51s)
epoch 1 step 650/9247 loss 3.7391 (last 50 steps took 16.49s)
epoch 1 step 700/9247 loss 3.7273 (last 50 steps took 16.44s)
epoch 1 step 750/9247 loss 3.7185 (last 50 steps took 16.41s)
epoch 1 step 800/9247 loss 3.7100 (last 50 steps took 16.36s)
epoch 1 s

## 7) Inference (Greedy / Beam) & Optional Attention Visualization

In [ ]:
# Reload best (optional)
if ckpt_path.exists():
    st = torch.load(ckpt_path, map_location=device)
    encoder.load_state_dict(st['encoder'])
    decoder.load_state_dict(st['decoder'])
    print('Loaded checkpoint:', ckpt_path)


def show_attention_on_image(img_pil, alphas, grid_hw):
    # alphas: list of (T=H*W) tensors for each step; we take last step for demo
    H,W = grid_hw
    if not alphas:
        plt.figure(figsize=(4,4)); plt.imshow(img_pil); plt.axis('off'); return
    att = alphas[-1][0].detach().cpu().numpy().reshape(H,W)
    att = (att - att.min())/(att.max()-att.min()+1e-8)
    plt.figure(figsize=(4,4))
    plt.imshow(img_pil)
    plt.imshow(att, cmap='jet', alpha=0.35)
    plt.axis('off')


def generate_caption(image_path: str, mode=None, beam_size=None, show=True):
    mode = mode or CFG['decode']
    beam_size = beam_size or CFG['beam_size']
    tfm = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.CenterCrop((224,224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ])
    img = Image.open(image_path).convert('RGB')
    with torch.no_grad():
        x = tfm(img).unsqueeze(0).to(device)
        bos = vocab.stoi[SPECIAL_TOKENS['bos']]
        eos = vocab.stoi[SPECIAL_TOKENS['eos']]
        if CFG['use_spatial_attention']:
            feats, grid_hw = encoder(x)
            if mode=='beam':
                ids = decoder.beam_search(feats, bos, eos, beam=beam_size, max_len=CFG['max_len'])
                toks = vocab.denumericalize(ids)
                alphas=None
            else:
                ids, alphas = decoder.greedy_decode(feats, bos, eos, max_len=CFG['max_len'])
                toks = vocab.denumericalize(ids[0])
        else:
            feat = encoder(x)
            ids = decoder.greedy_decode(feat, bos, eos, max_len=CFG['max_len'])[0]
            toks = vocab.denumericalize(ids)
            grid_hw=None; alphas=None
    if show:
        if CFG['use_spatial_attention'] and alphas is not None:
            show_attention_on_image(img, alphas, grid_hw)
        else:
            plt.figure(figsize=(4,4)); plt.imshow(img); plt.axis('off')
        plt.title('Predicted: ' + ' '.join(toks)); plt.show()
    return toks

# --- New logic to pick random images and generate captions ---

# 1. Define a list of image paths from val_ds
all_val_image_paths = [pair[0] for pair in val_ds.pairs]

# 2. Select 3-5 random image paths
num_random_images = random.randint(3, 5) # Select between 3 and 5 random images
random_image_paths = random.sample(all_val_image_paths, num_random_images)

print(f"Generating captions for {len(random_image_paths)} random validation images...")

# 3. Loop through the selected random image paths and call generate_caption
for i, img_path in enumerate(random_image_paths):
    print(f"\n--- Generating caption for image {i+1}/{len(random_image_paths)}: {img_path.name} ---")
    generate_caption(str(img_path), mode='beam', beam_size=5, show=True)
